# Act 1 — Tracing with MLflow

**Goal:** Instrument ZomatoBot so every LLM call is automatically captured as a trace in MLflow.

### Before you start
- MLflow server running: `mlflow server --host 127.0.0.1 --port 5000`
- `.env` file filled in (copy from `.env.example`)
- UI open at http://127.0.0.1:5000

---
## Setup

In [1]:
import os
import mlflow
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Session4 / Act 1 - Tracing")

# One line — automatically traces every OpenAI API call
mlflow.openai.autolog()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY", "ollama"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

SYSTEM_PROMPT = (
    "You are ZomatoBot, a helpful restaurant assistant for Indian cities. "
    "Answer in 2-3 sentences. Be polite and mention specific places when you can."
)

def call_llm(messages: list) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=100,
    )
    return response.choices[0].message.content

print(f"Model  : {MODEL}")
print(f"MLflow : http://127.0.0.1:5000")

2026/04/10 12:38:51 INFO mlflow.tracking.fluent: Experiment with name 'Session4 / Act 1 - Tracing' does not exist. Creating a new experiment.


Model  : llama3.2:1b
MLflow : http://127.0.0.1:5000


---
## Part A — Automatic tracing with `autolog()`

No code changes to your app. Just call it and traces appear.

In [2]:
SYSTEM_PROMPT = (
    "You are ZomatoBot, a helpful restaurant assistant for Indian cities. "
    "Answer in 2-3 sentences. Be polite and mention specific places when you can."
)

def zomato_bot(query: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": query},
        ],
        max_tokens=120,
    )
    return response.choices[0].message.content

# Each call below appears as a separate trace in the MLflow UI
print(zomato_bot("Best biryani near Bandra, Mumbai?"))
print("---")
print(zomato_bot("My order arrived cold. I want a refund!"))

I'd highly recommend checking out Karie's Masala Restaurant at the Andheri (Bandra) location, serving some of the best Hyderabadi biryani in town! Their unique blend of spices and tender biryani are sure to satisfy your cravings. You can easily find them near the Andheri Station Bus Stop on Mulund Road.
---
I'm truly sorry to hear that your order was delivered out of temperature control. At Zomato, we're committed to ensuring the food reaches our customers at its optimal freshness, so please accept our sincerest apologies for this inconvenience. If you'd like, I can assist you in initiating a refund and providing guidance on what you can do next.


[Trace(trace_id=tr-c141497a6eaca1d6d102e15dd2c430e6), Trace(trace_id=tr-f12827213e7ff4c69736639a26f7d62f)]

**👉 Open http://127.0.0.1:5000 → Traces tab**  
You should see 2 traces. Click one to see the full prompt, response, latency, and token count.

---
## Part B — Manual tracing with `@mlflow.trace`

Use this when you want to group multiple steps into one named trace,
or add business-level context (user ID, session, etc.).

In [3]:
@mlflow.trace(name="zomato_bot", attributes={"version": "v1"})
def zomato_bot_traced(query: str, user_id: str = "anonymous") -> str:
    # Tag the trace with session metadata
    mlflow.update_current_trace(
        tags={"user_id": user_id, "query_type": "recommendation"}
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": query},
        ],
        max_tokens=120,
    )
    return response.choices[0].message.content

result = zomato_bot_traced(
    "Cheap veg thali under Rs.100 in Pune?",
    user_id="user-42"
)
print(result)

If you're looking for a delicious and affordable veg thali in Pune, I'd recommend visiting the popular Eat.com to explore various options that start from around ₹60-₹80 per person. Their "Veg Thali" option at various eateries like Mewar, Navale Road's 'Dilip Buildwors', or even 'Udupi Lakshmi' in Koregaon Park can fit your budget of Rs. 100. Be sure to check reviews and wait times before ordering to ensure the best experience!


Trace(trace_id=tr-a0a12077b1865471d602fd3a8a1f6d01)

**👉 Back in the Traces tab** — the new trace has `user_id` and `query_type` tags visible in the detail view.

---
## Key takeaways

| | `autolog()` | `@mlflow.trace` |
|---|---|---|
| Setup effort | Zero | One decorator |
| What's captured | LLM call only | Your full function + LLM call |
| Custom tags/metadata | ❌ | ✅ |
| Use when | Quick instrumentation | Production observability |

➡️ **Next:** [02_evaluation.ipynb](02_evaluation.ipynb) — run structured evaluation on a golden dataset